# Train the waste-material detector on a free GPU

1. **Runtime -> Change runtime type -> T4 GPU**
2. On your PC, zip your prepared `data` folder (it must contain `images/` and `labels/`) as `data.zip`.
   Upload it to Google Drive (recommended) or straight into this notebook.
3. Run the cells top to bottom. At the end `best.pt` downloads - put it in your project's `models/` folder.

In [1]:
!nvidia-smi | head -n 15
!pip -q install ultralytics

'head' is not recognized as an internal or external command,
operable program or batch file.


## Get the data (choose ONE option)

In [2]:
# Option A - from Google Drive (put data.zip at the top level of My Drive)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/dataset
!unzip -q -o "/content/drive/MyDrive/data.zip" -d /content/dataset

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Option B - upload directly (slow for big datasets). Run this INSTEAD of option A.
# from google.colab import files
# up = files.upload()            # choose data.zip
# !mkdir -p /content/dataset
# !unzip -q -o data.zip -d /content/dataset

In [ ]:
# Find the folder that holds images/ and labels/ (zips sometimes add an extra level)
import os
root = None
for dp, dn, fn in os.walk('/content/dataset'):
    if 'images' in dn and 'labels' in dn:
        root = dp
        break
print('dataset root:', root)
assert root, 'could not find images/ and labels/ inside the zip'

In [ ]:
import yaml
names = ['cardboard', 'glass', 'metal', 'organic', 'other', 'paper', 'plastic', 'bin']   # MUST match configs/class_map.yaml
cfg = {'path': root, 'train': 'images/train', 'val': 'images/val',
       'names': {i: n for i, n in enumerate(names)}}
if os.path.isdir(os.path.join(root, 'images', 'test')):
    cfg['test'] = 'images/test'
yaml.safe_dump(cfg, open('/content/data.yaml', 'w'), sort_keys=False)
print(open('/content/data.yaml').read())

## Train
`yolov8s.pt` is a good balance. Use `yolov8n.pt` if you hit memory errors. 50 epochs is a sensible first run (about 30-60 min on a T4 for a few thousand images).

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(data='/content/data.yaml', epochs=50, imgsz=640, batch=16, name='baseline', project='/content/runs', exist_ok=True)

In [ ]:
metrics = YOLO('/content/runs/baseline/weights/best.pt').val(data='/content/data.yaml')
print('mAP@50     :', round(metrics.box.map50, 3))
print('mAP@50-95  :', round(metrics.box.map, 3))
for k, ci in enumerate(metrics.box.ap_class_index):
    print(f'{metrics.names[int(ci)]:<10} P={metrics.box.p[k]:.3f}  R={metrics.box.r[k]:.3f}  mAP50={metrics.box.ap50[k]:.3f}')

## Download the result
Save `best.pt` into `smart-waste-monitoring/models/best.pt`. Also keep the confusion matrix picture for your README.

In [ ]:
from google.colab import files
files.download('/content/runs/baseline/weights/best.pt')
import glob
for f in glob.glob('/content/runs/baseline/confusion_matrix_normalized.png') + glob.glob('/content/runs/baseline/results.png'):
    files.download(f)